**What this notebook is, and what it needs**

- It shows the classes of the `flx` package on the example data in `notebooks/example-dataset`: 100 synthetic
  fingerprint images (10 fingers with 10 images each) made with the SFinGe generator, with the minutiae that
  the generator wrote (`.ist` files).
- It is **not** the route for training on your own images. That route is `docs/training-guide.md` with the
  scripts in `scripts/`. For real images the minutiae come from `scripts/extract_minutiae.py` (Step 4 of
  that guide) as `.mnt` files and are read with `FingerFlowMinutiaMapLoader`.
- It needs the training environment of `docs/installation.md` with the optional tools (`jupyter`). Start
  Jupyter in the root directory of the repository or in `notebooks/`.
- It is stored without outputs. Its cells except the last one were run once on a CPU for the documents in
  `docs/`, with the model of the training tutorial after 2 epochs.
- **The last cell (the DET curve) is known not to run** with the packages that the optional tools install
  today: `flx.visualization` imports `tikzplotlib`, and `tikzplotlib` 0.10.1 cannot be imported together
  with matplotlib 3.11.2 (`ImportError: cannot import name 'common_texification'`). An older matplotlib was
  not tried. The cells before it do not need `tikzplotlib`.
- The model must exist first: run `model_training_tutorial.ipynb`, which writes `example-model/` in the
  root directory of the repository.

In this tutorial we will learn to:
- Load a previously trained model
- Extract DeepPrint features from fingerprint images
- Evaluate the performance of the extracted fixed-length representations

## Embedding extraction

After training the model, we can extract the DeepPrint features for the fingerprint images. This is done by calling the `extract` method of the `DeepPrintExtractor` class.

In [ ]:
import os
from pathlib import Path

from flx.extractor.fixed_length_extractor import get_DeepPrint_Tex, get_DeepPrint_TexMinu, DeepPrintExtractor

# The root directory of the repository. This works when Jupyter was started
# in the repository root or in its notebooks/ directory.
REPO = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
assert (REPO / "notebooks" / "example-dataset").is_dir(), (
    f"Start Jupyter in the root directory of the repository or in notebooks/ (it runs in {Path.cwd()})"
)

# The model must be built exactly as it was trained: the same variant, the same
# number of values per branch (num_dims) and the same number of training subjects.
# These are the values of the model that model_training_tutorial.ipynb trains.
# (Loading the pre-trained model of the paper was not tried; see docs/paper.md.)
extractor: DeepPrintExtractor = get_DeepPrint_TexMinu(num_training_subjects=5, num_dims=256)

# The directory with the model parameters, written by model_training_tutorial.ipynb
MODEL_DIR: str = str(REPO / "example-model")
# load_best_model only prints a message when the file is missing and then goes on
# with an untrained model, so check it here
assert (Path(MODEL_DIR) / "best_model.pyt").is_file(), (
    f"No model in {MODEL_DIR}: run model_training_tutorial.ipynb first"
)
extractor.load_best_model(MODEL_DIR)

Now we need to specify the dataset, for which we want to extract the embeddings

In [ ]:
import os

from flx.data.dataset import *
from flx.data.image_loader import SFingeLoader
from flx.data.transformed_image_loader import TransformedImageLoader
from flx.image_processing.binarization import LazilyAllocatedBinarizer
from flx.data.image_helpers import pad_and_resize_to_deepprint_input_size

DATASET_PATH: str = str(REPO / "notebooks" / "example-dataset")

# We will use the SFingeLoader to load the images from the dataset
image_loader = TransformedImageLoader(
        images=SFingeLoader(DATASET_PATH),
        poses=None,
        transforms=[
            LazilyAllocatedBinarizer(5.0),
            pad_and_resize_to_deepprint_input_size,
        ],
    )

image_dataset: Dataset = Dataset(image_loader, image_loader.ids)

# Texture and minutia embeddings. The second is None for the texture-only variants
texture_embeddings, minutia_embeddings = extractor.extract(image_dataset)

## Benchmarking

To evaluate the embeddings, we want to run a benchmark on them. For this, we must first specify the type of benchmark, and which comparisons should be run.

In [ ]:
from flx.scripts.generate_benchmarks import create_verification_benchmark

NUM_IMPRESSIONS_PER_SUBJECT = 10
benchmark = create_verification_benchmark(
    subjects=list(range(image_dataset.num_subjects)),
    impressions_per_subject=list(range(NUM_IMPRESSIONS_PER_SUBJECT))
)

Now we can run the benchmark. To do this, we must first specify the matcher (in our case cosine similarity of the embeddings)

The equal error rate that this prints means nothing: the benchmark runs on all 100 example images, 50 of which are the training images of the model, and the model was trained for a few steps only.

In [ ]:
from flx.benchmarks.matchers import CosineSimilarityMatcher
from flx.data.embedding_loader import EmbeddingLoader

# We concatenate texture and minutia embedding vectors
embeddings = EmbeddingLoader.combine(texture_embeddings, minutia_embeddings)
matcher = CosineSimilarityMatcher(EmbeddingLoader.combine(texture_embeddings, minutia_embeddings))

results = benchmark.run(matcher)

print(f"Equal-Error-Rate: {results.get_equal_error_rate()}")

To visualize the results, we can plot a DET curve. (Do not wonder if it is empty, probably the model is not trained enough. Take a look at the EER instead.)

**This cell is known not to run with current package versions**, see the note at the top of the notebook. It needs a `tikzplotlib` that can be imported with the installed matplotlib.

In [ ]:
from flx.visualization.plot_DET_curve import plot_verification_results

figure_path = "DET_curve"

# Lists are used to allow for multiple models to be plotted in the same figure
plot_verification_results(figure_path, results=[results], model_labels=["DeepPrint_TexMinu"], plot_title="example-dataset - verification")